Code: https://github.com/zhurleonid/corporate-risk-management
(`homework_5/hw5_code.ipynb`)

The write-up is below and the code is appended at the end. Part (a), the
closed-form price and the two deltas, is derived by hand and attached in front.

# Scenario analysis of a delta-hedged spread-squared option

## Set-up

Long one unit of the option paying $(S_T^{(1)} - S_T^{(2)})^2$, short $h^{(1)}$
and $h^{(2)}$ shares, with the hedge held fixed over five days, $\Delta = 1/252$:

$$V_t = V(t, S_t^{(1)}, S_t^{(2)}; \sigma_t, \rho_t) - h^{(1)}_t S_t^{(1)} - h^{(2)}_t S_t^{(2)}, \qquad
V_{t+5\Delta} = V(t+5\Delta, S^{(1)}_{t+5\Delta}, S^{(2)}_{t+5\Delta}; \sigma_{t+5\Delta}, \rho_{t+5\Delta}) - h^{(1)}_t S^{(1)}_{t+5\Delta} - h^{(2)}_t S^{(2)}_{t+5\Delta}.$$

Parameters: $r = 1.32\%$, $\sigma_{1,0} = 22.14\%$, $\sigma_{2,0} = 30.36\%$,
$\rho_0 = 0.237$, $T = 0.25$, $S_0^{(1)} = 158.12$, $S_0^{(2)} = 170.33$. The drifts
$\mu_1, \mu_2$ play no part: the price is risk neutral and the scenarios fix the
log returns directly.

## (a) The price

> **By hand, attached:** under $\mathbb{Q}$ each $S_T^{(i)}$ is lognormal, so the three
> terms of $(S^{(1)}_T - S^{(2)}_T)^2$ are expectations of exponentials of normals
> with variances $4\sigma_1^2\tau$, $(\sigma_1^2 + \sigma_2^2 + 2\rho\sigma_1\sigma_2)\tau$
> and $4\sigma_2^2\tau$. Discounting by $e^{-r\tau}$ gives

$$V(t,s_1,s_2;\sigma,\rho)=e^{r(T-t)}\Big[s_1^2e^{\sigma_1^2(T-t)}-2s_1s_2e^{\sigma_1\sigma_2\rho(T-t)}+s_2^2e^{\sigma_2^2(T-t)}\Big],$$

$$h^{(1)} = 2e^{r\tau}\big[s_1 e^{\sigma_1^2\tau} - s_2 e^{\sigma_1\sigma_2\rho\tau}\big], \qquad
h^{(2)} = 2e^{r\tau}\big[s_2 e^{\sigma_2^2\tau} - s_1 e^{\sigma_1\sigma_2\rho\tau}\big].$$

At $t = 0$:

$$h^{(1)} = -21.9525, \qquad h^{(2)} = 31.2019, \qquad V(0,\cdot) = 921.74, \qquad V_0 = -921.74.$$

$V$ is homogeneous of degree 2 in $(s_1, s_2)$, so $s_1 h^{(1)} + s_2 h^{(2)} = 2V$
and the hedged portfolio is worth exactly $-V$. The hedge is long 22 shares of
stock 1 and short 31 of stock 2. Only $149.58$ of the option value is the
discounted squared spread $e^{rT}(S_0^{(1)} - S_0^{(2)})^2$; the remaining $772.16$
is, to first order, $e^{rT}\,T\,\big(s_1^2\sigma_1^2 - 2\rho\sigma_1\sigma_2 s_1 s_2 + s_2^2\sigma_2^2\big) = 762.91$,
the variance of the spread over $[0, T]$. The position is essentially long the
variance of the spread, which is what drives the results below.

## Pseudocode

```
inputs: r, sigma1_0, sigma2_0, rho_0, T, S1_0, S2_0, Delta
V(t, s1, s2; sig1, sig2, rho) = the closed form from (a), tau = T - t

hedge at t = 0:   h1 = dV/ds1 (0, S1_0, S2_0; sigma_0, rho_0),  h2 = dV/ds2 (...)
portfolio:        V_0 = V(0, S1_0, S2_0; sigma_0, rho_0) - h1*S1_0 - h2*S2_0

grids:  X in {-0.20, -0.10, -0.05, 0.05, 0.10, 0.20}      (each stock)
        k in {0.5, 0.75, 1.25, 1.5, 1.75, 2}              (each vol ratio)
        d_rho in {-0.5, 0, 0.5}
weights: wX(|X|) = 0.5 / 0.75 / 1 for |X| = 0.20 / 0.10 / 0.05
         wK(k)   = 0.5 for k in {0.5, 2}, 1.25 for {0.75, 1.75}, 0.75 for {1.25, 1.5}
         wR(|d_rho|) = 0.5 for 0, 1.25 for 0.5

for each (X1, X2, k1, k2, d_rho) in X x X x K x K x R        (N = 3888):
    S1 = S1_0*exp(X1),  S2 = S2_0*exp(X2)
    sig1 = k1*sigma1_0,  sig2 = k2*sigma2_0,  rho = rho_0 + d_rho
    V_5 = V(5*Delta, S1, S2; sig1, sig2, rho) - h1*S1 - h2*S2     hedge held fixed
    l_n = -(V_5 - V_0)
    w_n = wX(|X1|) * wX(|X2|) * wK(k1) * wK(k2) * wR(|d_rho|)
    store (scenario, l_n, w_n, w_n*l_n)

(b) report max_n l_n and its scenario
(c) report max_n w_n*l_n, its scenario, and l_n there
```

The sheet's example weight checks out: $X^{(1)} = -0.1$, $X^{(2)} = 0.2$,
$k_1 = 2$, $k_2 = 0.5$, $\delta\rho = -0.5$ gives $w = 15/128$.

## (b) Worst-case scenario

$$\rho(L_{t+5\Delta}) = \max_n l_n = 1000.36$$

attained at

$$X^{(1)} = X^{(2)} = +0.20, \qquad \frac{\sigma^{(1)}_{t+5\Delta}}{\sigma^{(1)}_t} = \frac{\sigma^{(2)}_{t+5\Delta}}{\sigma^{(2)}_t} = 0.5, \qquad \rho_{t+5\Delta} - \rho_t = +0.5.$$

The loss exceeds the option's whole value at $t = 0$. The eight largest losses:

$$\begin{array}{r r r r r r r}
\hline
X^{(1)} & X^{(2)} & k_1 & k_2 & \delta\rho & \ell_n & w_n \\
\hline
+0.20 & +0.20 & 0.50 & 0.50 & +0.5 & 1000.36 & 5/64 \\
+0.20 & +0.20 & 0.75 & 0.50 & +0.5 & 983.14 & 25/128 \\
+0.20 & +0.20 & 0.75 & 0.75 & +0.5 & 866.08 & 125/256 \\
+0.10 & +0.10 & 0.50 & 0.50 & +0.5 & 845.82 & 45/256 \\
+0.20 & +0.20 & 0.50 & 0.50 & \;\;0.0 & 844.20 & 1/32 \\
+0.10 & +0.10 & 0.75 & 0.50 & +0.5 & 831.73 & 0.4395 \\
+0.20 & +0.20 & 0.50 & 0.75 & +0.5 & 825.35 & 25/128 \\
+0.20 & +0.20 & 1.25 & 0.50 & +0.5 & 787.04 & 15/128 \\
\hline
\end{array}$$

**Why this scenario.** Splitting the loss into the price move at the old
parameters and the parameter change at the new prices gives $47.59 + 952.77$.
The price move itself barely matters: the hedge removes the first-order
exposure, and the option is convex in the spread, so a pure price move at
fixed parameters loses at most $66$ on the whole grid and gains up to $4390$
when the two stocks move apart. Almost all of the loss is the fall in spread
variance: both volatilities halve and the correlation rises to $0.737$, which
cuts $s_1^2\sigma_1^2 - 2\rho\sigma_1\sigma_2 s_1 s_2 + s_2^2\sigma_2^2$ by
the most the grid allows. Both stocks up 20% comes in because this variance
exposure scales with $s_1^2$ and $s_2^2$, so higher prices make the same
parameter shock cost more, while the spread itself hardly changes when the two
move together. The variance approximation of the parameter part is $941.52$,
within 1.2% of the exact $952.77$.

## (c) Weighted scenario

$$\rho(L_{t+5\Delta}) = \max_n w_n l_n = 1313.79$$

attained at

$$X^{(1)} = X^{(2)} = +0.05, \qquad \frac{\sigma^{(1)}_{t+5\Delta}}{\sigma^{(1)}_t} = \frac{\sigma^{(2)}_{t+5\Delta}}{\sigma^{(2)}_t} = 0.75, \qquad \rho_{t+5\Delta} - \rho_t = +0.5,$$

with weight $w_n = 1 \times 1 \times 1.25 \times 1.25 \times 1.25 = 125/64 = 1.9531$.
**The portfolio loss at this scenario is $\ell_n = 672.66$**, which is 67.2% of
the unweighted worst case. The eight largest weighted losses:

$$\begin{array}{r r r r r r r r}
\hline
X^{(1)} & X^{(2)} & k_1 & k_2 & \delta\rho & \ell_n & w_n & w_n\ell_n \\
\hline
+0.05 & +0.05 & 0.75 & 0.75 & +0.5 & 672.66 & 1.9531 & 1313.79 \\
-0.05 & -0.05 & 0.75 & 0.75 & +0.5 & 550.52 & 1.9531 & 1075.23 \\
+0.10 & +0.05 & 0.75 & 0.75 & +0.5 & 633.31 & 1.4648 & 927.70 \\
+0.05 & +0.10 & 0.75 & 0.75 & +0.5 & 617.26 & 1.4648 & 904.19 \\
+0.10 & +0.10 & 0.75 & 0.75 & +0.5 & 735.89 & 1.0986 & 808.47 \\
+0.05 & +0.05 & 1.75 & 0.75 & +0.5 & 390.88 & 1.9531 & 763.43 \\
+0.05 & +0.05 & 1.25 & 0.75 & +0.5 & 613.70 & 1.1719 & 719.18 \\
-0.10 & -0.05 & 0.75 & 0.75 & +0.5 & 485.07 & 1.4648 & 710.55 \\
\hline
\end{array}$$

The weights move the measure from the extreme corner of the grid to a
moderate scenario: a small common move up, volatilities down a quarter,
correlation up. $125/64$ is the largest weight on the grid, shared by 32
scenarios, and the (c) scenario is the one of those 32 with the largest loss.
The weighting discounts the (b) scenario by $5/64$ because halving both
volatilities and a 20% move in each stock are all treated as unlikely over
five days, whereas $-25\%$ in volatility and $+0.5$ in correlation carry weight
above one. The direction of the loss is the same as in (b): $66.31$ from the
price move and $606.35$ from the parameter change (variance approximation
$599.32$). Note that $w_n\ell_n = 1313.79$ is larger than any actual loss on the
grid, since the weight exceeds one; the number to compare with (b) is the loss
at the scenario, $672.66$ against $1000.36$.